# 09. Validación de extracción clínica con LLM local

Este notebook evalúa la parte de extracción de texto libre que se usa en la aplicación. El objetivo no es entrenar un modelo nuevo ni volver a decidir el clasificador final, sino comprobar si distintos modelos locales de Ollama son capaces de convertir una historia clínica breve de urgencias en un `VectorClinico` estructurado.

La prueba se hace con casos sintéticos, escritos con estilo de triaje y basados en presentaciones clínicas habituales de urgencias. No contienen datos reales de pacientes. Se han usado como referencia general materiales públicos sobre triaje ESI y casos docentes de urgencias, especialmente el manual ESI, escenarios docentes de dolor torácico y ejemplos de evaluación inicial en urgencias.

Fuentes consultadas como contexto:

- Emergency Severity Index, A Triage Tool for Emergency Department Care, versión 4.
- UCSF Hospital Handbook: Chest Pain.
- International Emergency Medicine Education Project: Chest Pain.
- WHO emergency triage case scenarios.

## Planteamiento

En el sistema final el LLM no decide el nivel de triaje. Su tarea es extraer datos clínicos desde una narrativa libre. Después, el resultado pasa por tres capas de control:

1. `VectorClinico`, que valida tipos y rangos con Pydantic.
2. `normalizar_vector_clinico`, que limpia sinónimos frecuentes y evita duplicados.
3. `validar_vector_clinico`, que avisa de incoherencias o datos mencionados pero no extraídos.

La comparación se centra en cuatro modelos locales disponibles en Ollama:

- `llama3.1:8b-instruct-q4_K_M`, como modelo principal equilibrado.
- `llama3.2`, como modelo más rápido.
- `qwen2.5:7b`, como alternativa similar.
- `qwen2.5:14b`, como alternativa más costosa.

La métrica usada es sencilla: porcentaje de campos esperados extraídos correctamente en 15 casos clínicos controlados. También se mide tiempo de respuesta y número de alertas del validador.

In [1]:
from __future__ import annotations

import json
import subprocess
import time
from pathlib import Path
from typing import Any

import pandas as pd

from triaje_ia.config import REPORTS_DIR
from triaje_ia.llm.extractor import extraer_vector_clinico
from triaje_ia.llm.validator import validar_vector_clinico

RANDOM_STATE = 42

OUT_DIR = REPORTS_DIR / "llm_extraction_validation"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODELOS_OLLAMA = [
    "llama3.1:8b-instruct-q4_K_M",
    "llama3.2",
    "qwen2.5:7b",
    "qwen2.5:14b",
]

GUARDAR_RESULTADOS = True

print(f"Directorio de salida: {OUT_DIR}")
print(f"Modelos a evaluar: {len(MODELOS_OLLAMA)}")

Directorio de salida: C:\Users\CARLOS\triaje-ia-tfg\reports\llm_extraction_validation
Modelos a evaluar: 4


## Comprobación de modelos locales

Antes de lanzar la evaluación se comprueba que Ollama tiene descargados los modelos necesarios. Si falta alguno, el notebook se detiene con un mensaje claro. Esto evita interpretar como fallo metodológico lo que en realidad sería un problema de entorno.

In [2]:
def modelos_ollama_disponibles() -> set[str]:
    result = subprocess.run(
        ["ollama", "list"],
        capture_output=True,
        text=True,
        check=True,
    )
    lineas = result.stdout.splitlines()[1:]
    modelos = {linea.split()[0] for linea in lineas if linea.strip()}
    aliases_latest = {
        modelo.removesuffix(":latest")
        for modelo in modelos
        if modelo.endswith(":latest")
    }
    return modelos | aliases_latest


disponibles = modelos_ollama_disponibles()
faltantes = [m for m in MODELOS_OLLAMA if m not in disponibles]

assert not faltantes, (
    "Faltan modelos de Ollama. Ejecutar antes: "
    + "; ".join(f"ollama pull {m}" for m in faltantes)
)

print("Modelos disponibles:")
for modelo in MODELOS_OLLAMA:
    print(f"- {modelo}")

Modelos disponibles:
- llama3.1:8b-instruct-q4_K_M
- llama3.2
- qwen2.5:7b
- qwen2.5:14b


## Casos clínicos sintéticos

Los casos cubren situaciones frecuentes de triaje: dolor torácico, disnea, focalidad neurológica, dolor abdominal, fiebre pediátrica, trauma, alergia, hipoglucemia, ansiedad, constantes contradictorias, ausencia de constantes, medicación crónica y texto poco específico.

Cada caso incluye una lista de campos esperados. No se pretende evaluar todo el razonamiento clínico, solo si el LLM extrae correctamente la información que aparece en la historia.

In [3]:
CASOS = [
    {
        "case_id": "C01_dolor_toracico",
        "comentario": "Dolor torácico con constantes completas y medicación habitual.",
        "narrativa": (
            "Varón de 58 años que acude a triaje por dolor torácico opresivo "
            "desde hace 45 minutos, con sudoración fría y náuseas. Antecedente "
            "de hipertensión arterial en tratamiento con enalapril. TA 160/95, "
            "FC 104 lpm, FR 18 rpm, SatO2 97%, temperatura 36.7. Dolor 8/10."
        ),
        "expected": {
            "edad": 58,
            "sexo": "M",
            "presion_sistolica": 160,
            "presion_diastolica": 95,
            "frecuencia_cardiaca": 104,
            "frecuencia_respiratoria": 18,
            "saturacion_oxigeno": 97.0,
            "temperatura": 36.7,
            "nivel_dolor": 8,
            "sintomas_presentes": ["chest pain", "nausea"],
            "medicacion_habitual": ["ace inhibitors"],
        },
    },
    {
        "case_id": "C02_disnea_epoc",
        "comentario": "Disnea en paciente respiratoria crónica con hipoxemia.",
        "narrativa": (
            "Mujer de 74 años con EPOC conocida. Consulta por disnea progresiva "
            "desde esta mañana, tos y aumento de expectoración. SatO2 86% basal, "
            "FR 30 rpm, FC 118 lpm, TA 145/80, temperatura 37.2. Usa salbutamol "
            "inhalado en domicilio."
        ),
        "expected": {
            "edad": 74,
            "sexo": "F",
            "presion_sistolica": 145,
            "presion_diastolica": 80,
            "frecuencia_cardiaca": 118,
            "frecuencia_respiratoria": 30,
            "saturacion_oxigeno": 86.0,
            "temperatura": 37.2,
            "sintomas_presentes": ["dyspnea", "cough"],
        },
    },
    {
        "case_id": "C03_focalidad_neurologica",
        "comentario": "Inicio brusco de focalidad neurológica y anticoagulación.",
        "narrativa": (
            "Hombre de 67 años traído por su familia por inicio brusco hace 30 "
            "minutos de desviación de comisura, debilidad en brazo derecho y "
            "dificultad para hablar. Antecedente de fibrilación auricular en "
            "tratamiento con acenocumarol. TA 180/100, FC 92 irregular, SatO2 96%."
        ),
        "expected": {
            "edad": 67,
            "sexo": "M",
            "presion_sistolica": 180,
            "presion_diastolica": 100,
            "frecuencia_cardiaca": 92,
            "saturacion_oxigeno": 96.0,
            "medicacion_habitual": ["vitamin k antagonists"],
        },
    },
    {
        "case_id": "C04_dolor_abdominal",
        "comentario": "Dolor abdominal localizado con náuseas, vómitos y dolor medido.",
        "narrativa": (
            "Mujer de 35 años con dolor abdominal en fosa iliaca derecha de 10 "
            "horas de evolución, acompañado de náuseas y dos vómitos. Sin "
            "antecedentes relevantes. TA 118/70, FC 96, SatO2 99%, temperatura "
            "37.9. Refiere dolor 7/10."
        ),
        "expected": {
            "edad": 35,
            "sexo": "F",
            "presion_sistolica": 118,
            "presion_diastolica": 70,
            "frecuencia_cardiaca": 96,
            "saturacion_oxigeno": 99.0,
            "temperatura": 37.9,
            "nivel_dolor": 7,
            "sintomas_presentes": ["abdominal pain", "nausea", "vomiting"],
        },
    },
    {
        "case_id": "C05_fiebre_pediatrica",
        "comentario": "Paciente pediátrico con fiebre alta y vómitos.",
        "narrativa": (
            "Niño de 6 años que acude con su madre por fiebre de 39.5 desde ayer, "
            "decaimiento y vómitos. No toma medicación habitual. FC 130, FR 24, "
            "SatO2 98%, TA 100/60."
        ),
        "expected": {
            "edad": 6,
            "presion_sistolica": 100,
            "presion_diastolica": 60,
            "frecuencia_cardiaca": 130,
            "frecuencia_respiratoria": 24,
            "saturacion_oxigeno": 98.0,
            "temperatura": 39.5,
            "sintomas_presentes": ["fever", "vomiting"],
        },
    },
    {
        "case_id": "C06_trauma_muneca",
        "comentario": "Trauma menor con dolor localizado y constantes normales.",
        "narrativa": (
            "Varón de 28 años tras caída en bicicleta. Presenta dolor intenso en "
            "muñeca izquierda y deformidad leve. Niega pérdida de conciencia. "
            "TA 125/78, FC 88, SatO2 99%, temperatura 36.4. Dolor 6/10."
        ),
        "expected": {
            "edad": 28,
            "sexo": "M",
            "presion_sistolica": 125,
            "presion_diastolica": 78,
            "frecuencia_cardiaca": 88,
            "saturacion_oxigeno": 99.0,
            "temperatura": 36.4,
            "nivel_dolor": 6,
            "sintomas_presentes": ["pain"],
        },
    },
    {
        "case_id": "C07_reaccion_alergica",
        "comentario": "Reacción alérgica sin compromiso respiratorio inicial.",
        "narrativa": (
            "Mujer de 42 años con urticaria generalizada y sensación de hinchazón "
            "en labios tras tomar amoxicilina. Niega disnea y no presenta estridor. "
            "TA 110/70, FC 105, FR 18, SatO2 98%."
        ),
        "expected": {
            "edad": 42,
            "sexo": "F",
            "presion_sistolica": 110,
            "presion_diastolica": 70,
            "frecuencia_cardiaca": 105,
            "frecuencia_respiratoria": 18,
            "saturacion_oxigeno": 98.0,
            "sintomas_presentes": ["urticaria", "lip swelling"],
        },
    },
    {
        "case_id": "C08_hipoglucemia",
        "comentario": "Alteración del nivel de conciencia en diabético con insulina.",
        "narrativa": (
            "Hombre de 71 años diabético tratado con insulina. Llega confuso, "
            "sudoroso y con dificultad para responder. Glucemia capilar 48. "
            "TA 135/75, FC 90, SatO2 97%, temperatura 36.1."
        ),
        "expected": {
            "edad": 71,
            "sexo": "M",
            "presion_sistolica": 135,
            "presion_diastolica": 75,
            "frecuencia_cardiaca": 90,
            "saturacion_oxigeno": 97.0,
            "temperatura": 36.1,
            "sintomas_presentes": ["altered mental status", "diaphoresis"],
            "medicacion_habitual": ["insulin analogues"],
        },
    },
    {
        "case_id": "C09_ansiedad_dolor_negado",
        "comentario": "Síntomas ansiosos con dolor torácico negado.",
        "narrativa": (
            "Mujer de 24 años con palpitaciones, temblor y sensación de falta de "
            "aire tras una discusión. Niega dolor torácico. TA 128/78, FC 115, "
            "FR 22, SatO2 100%, temperatura 36.6."
        ),
        "expected": {
            "edad": 24,
            "sexo": "F",
            "presion_sistolica": 128,
            "presion_diastolica": 78,
            "frecuencia_cardiaca": 115,
            "frecuencia_respiratoria": 22,
            "saturacion_oxigeno": 100.0,
            "temperatura": 36.6,
            "sintomas_presentes": ["palpitations", "tremor", "dyspnea"],
        },
    },
    {
        "case_id": "C10_tension_invertida",
        "comentario": "Constante posiblemente mal registrada, que debe conservarse y alertarse.",
        "narrativa": (
            "Varón de 80 años que consulta por mareo. Enfermería anota TA 70/120, "
            "FC 55, SatO2 95%, temperatura 36.0. Antecedente de cardiopatía."
        ),
        "expected": {
            "edad": 80,
            "sexo": "M",
            "presion_sistolica": 70,
            "presion_diastolica": 120,
            "frecuencia_cardiaca": 55,
            "saturacion_oxigeno": 95.0,
            "temperatura": 36.0,
            "sintomas_presentes": ["dizziness"],
        },
    },
    {
        "case_id": "C11_sin_constantes",
        "comentario": "Historia leve sin constantes registradas en el texto.",
        "narrativa": (
            "Mujer de 31 años que consulta por cefalea leve desde esta mañana. "
            "Sin vómitos, sin fiebre referida y sin medicación habitual."
        ),
        "expected": {
            "edad": 31,
            "sexo": "F",
            "sintomas_presentes": ["headache"],
        },
    },
    {
        "case_id": "C12_medicacion_compleja",
        "comentario": "Paciente pluripatológico con varias clases terapéuticas relevantes.",
        "narrativa": (
            "Hombre de 69 años con insuficiencia cardiaca. Toma furosemida, "
            "bisoprolol y sintrom. Acude por aumento de disnea y edemas en miembros "
            "inferiores. TA 150/85, FC 98, FR 26, SatO2 91%."
        ),
        "expected": {
            "edad": 69,
            "sexo": "M",
            "presion_sistolica": 150,
            "presion_diastolica": 85,
            "frecuencia_cardiaca": 98,
            "frecuencia_respiratoria": 26,
            "saturacion_oxigeno": 91.0,
            "sintomas_presentes": ["dyspnea", "edema"],
            "medicacion_habitual": [
                "loop diuretics",
                "beta blockers cardiac selective",
                "vitamin k antagonists",
            ],
        },
    },
    {
        "case_id": "C13_texto_vago",
        "comentario": "Narrativa poco específica donde el extractor no debe inventar síntomas graves.",
        "narrativa": (
            "Paciente mujer de 50 años que se encuentra mal desde ayer, muy "
            "cansada, sin explicar mejor el motivo. No refiere dolor. TA 122/76, "
            "FC 84, SatO2 98%, temperatura 36.8."
        ),
        "expected": {
            "edad": 50,
            "sexo": "F",
            "presion_sistolica": 122,
            "presion_diastolica": 76,
            "frecuencia_cardiaca": 84,
            "saturacion_oxigeno": 98.0,
            "temperatura": 36.8,
        },
    },
    {
        "case_id": "C14_afebril_con_fiebre",
        "comentario": "Contradicción entre lo referido por el paciente y la temperatura medida.",
        "narrativa": (
            "Varón de 45 años dice estar afebril en domicilio, pero en triaje se "
            "mide temperatura 39.1. Consulta por tos y dolor pleurítico. TA 130/80, "
            "FC 110, FR 24, SatO2 93%."
        ),
        "expected": {
            "edad": 45,
            "sexo": "M",
            "presion_sistolica": 130,
            "presion_diastolica": 80,
            "frecuencia_cardiaca": 110,
            "frecuencia_respiratoria": 24,
            "saturacion_oxigeno": 93.0,
            "temperatura": 39.1,
            "sintomas_presentes": ["cough", "chest pain"],
        },
    },
    {
        "case_id": "C15_dolor_sin_escala",
        "comentario": "Dolor descrito sin escala numérica, ?til para comprobar alertas.",
        "narrativa": (
            "Mujer de 62 años con dolor lumbar intenso irradiado a pierna izquierda. "
            "No se registra escala de dolor. TA 140/85, FC 78, SatO2 98%."
        ),
        "expected": {
            "edad": 62,
            "sexo": "F",
            "presion_sistolica": 140,
            "presion_diastolica": 85,
            "frecuencia_cardiaca": 78,
            "saturacion_oxigeno": 98.0,
            "sintomas_presentes": ["back pain"],
        },
    },
]

print(f"Casos definidos: {len(CASOS)}")
assert len(CASOS) == 15
assert all("expected" in c and "narrativa" in c for c in CASOS)

Casos definidos: 15


## Funciones de evaluación

La evaluación compara valores exactos en campos numéricos y de sexo/edad. En listas de síntomas y medicación no se exige igualdad completa, sino que aparezcan los términos esperados dentro de la lista extraída. Esto es más razonable porque distintos modelos pueden devolver sinónimos clínicos válidos.

In [4]:
CAMPOS_NUMERICOS = [
    "edad",
    "presion_sistolica",
    "presion_diastolica",
    "frecuencia_cardiaca",
    "frecuencia_respiratoria",
    "saturacion_oxigeno",
    "temperatura",
    "nivel_dolor",
]


def contiene_termino(lista: list[str], esperado: str) -> bool:
    texto = " | ".join(lista).lower()
    return esperado.lower() in texto


def comparar_campo(vector: Any, campo: str, esperado: Any) -> tuple[bool, Any]:
    obtenido = getattr(vector, campo)

    if campo in {"sintomas_presentes", "medicacion_habitual"}:
        ok = all(contiene_termino(obtenido, termino) for termino in esperado)
        return ok, obtenido

    if isinstance(esperado, float):
        ok = obtenido is not None and abs(float(obtenido) - esperado) < 0.05
        return ok, obtenido

    return obtenido == esperado, obtenido


def evaluar_caso(modelo: str, caso: dict[str, Any]) -> dict[str, Any]:
    inicio = time.perf_counter()
    base = {
        "modelo": modelo,
        "case_id": caso["case_id"],
        "comentario": caso["comentario"],
        "error": "",
    }

    try:
        vector = extraer_vector_clinico(caso["narrativa"], modelo=modelo)
        segundos = time.perf_counter() - inicio
        alertas = validar_vector_clinico(vector, caso["narrativa"])

        fallos = []
        aciertos = 0
        total = 0

        for campo, esperado in caso["expected"].items():
            total += 1
            ok, obtenido = comparar_campo(vector, campo, esperado)
            if ok:
                aciertos += 1
            else:
                fallos.append(
                    {
                        "campo": campo,
                        "esperado": esperado,
                        "obtenido": obtenido,
                    }
                )

        return {
            **base,
            "segundos": segundos,
            "campos_ok": aciertos,
            "campos_total": total,
            "accuracy_campos": aciertos / total if total else 0.0,
            "n_alertas": len(alertas),
            "alertas": [str(a) for a in alertas],
            "fallos": fallos,
            "vector": vector.model_dump(),
        }

    except Exception as exc:
        segundos = time.perf_counter() - inicio
        return {
            **base,
            "segundos": segundos,
            "campos_ok": 0,
            "campos_total": len(caso["expected"]),
            "accuracy_campos": 0.0,
            "n_alertas": 0,
            "alertas": [],
            "fallos": [],
            "vector": {},
            "error": f"{type(exc).__name__}: {exc}",
        }

## Ejecución de la comparación

Esta celda puede tardar varios minutos porque ejecuta 60 extracciones en total. Se mantiene como ejecución local para respetar la privacidad del texto y porque la aplicación final está pensada para funcionar con Ollama.

In [5]:
resultados = []

for modelo in MODELOS_OLLAMA:
    print(f"\nModelo: {modelo}")
    for caso in CASOS:
        fila = evaluar_caso(modelo, caso)
        resultados.append(fila)
        resumen = f"{fila['campos_ok']}/{fila['campos_total']}"
        tiempo = f"{fila['segundos']:.1f}s"
        estado = "ERROR" if fila["error"] else resumen
        print(f"  {caso['case_id']}: {estado} ({tiempo})")

results_df = pd.DataFrame(resultados)
results_df.head()

2026-07-05 12:03:02.339 | INFO     | triaje_ia.llm.extractor:extraer_vector_clinico:248 - Iniciando extracción con modelo 'llama3.1:8b-instruct-q4_K_M'



Modelo: llama3.1:8b-instruct-q4_K_M


2026-07-05 12:03:46.241 | SUCCESS  | triaje_ia.llm.extractor:extraer_vector_clinico:264 - Extraídos 2 síntomas
2026-07-05 12:03:46.243 | INFO     | triaje_ia.llm.extractor:extraer_vector_clinico:248 - Iniciando extracción con modelo 'llama3.1:8b-instruct-q4_K_M'


  C01_dolor_toracico: 11/11 (43.9s)


2026-07-05 12:03:50.658 | SUCCESS  | triaje_ia.llm.extractor:extraer_vector_clinico:264 - Extraídos 3 síntomas
2026-07-05 12:03:50.660 | INFO     | triaje_ia.llm.extractor:extraer_vector_clinico:248 - Iniciando extracción con modelo 'llama3.1:8b-instruct-q4_K_M'


...

2026-07-05 12:17:15.777 | SUCCESS  | triaje_ia.llm.extractor:extraer_vector_clinico:264 - Extraídos 3 síntomas
2026-07-05 12:17:15.779 | INFO     | triaje_ia.llm.extractor:extraer_vector_clinico:248 - Iniciando extracción con modelo 'qwen2.5:14b'


  C14_afebril_con_fiebre: 8/9 (34.9s)


2026-07-05 12:17:47.509 | SUCCESS  | triaje_ia.llm.extractor:extraer_vector_clinico:264 - Extraídos 2 síntomas


  C15_dolor_sin_escala: 7/7 (31.7s)


,modelo,case_id,comentario,error,segundos,campos_ok,campos_total,accuracy_campos,n_alertas,alertas,fallos,vector
0,llama3.1:8b-instruct-q4_K_M,C01_dolor_toracico,Dolor torácico con constantes completas y medi...,,43.902499,11,11,1.000000,0,[],[],"{'edad': 58, 'sexo': 'M', 'sintomas_presentes'..."
1,llama3.1:8b-instruct-q4_K_M,C02_disnea_epoc,Disnea en paciente respiratoria crónica con hi...,,4.416636,9,9,1.000000,0,[],[],"{'edad': 74, 'sexo': 'F', 'sintomas_presentes'..."
2,llama3.1:8b-instruct-q4_K_M,C03_focalidad_neurologica,Inicio brusco de focalidad neurológica y antic...,,3.611372,6,7,0.857143,0,[],"[{'campo': 'medicacion_habitual', 'esperado': ...","{'edad': 67, 'sexo': 'M', 'sintomas_presentes'..."
3,llama3.1:8b-instruct-q4_K_M,C04_dolor_abdominal,"Dolor abdominal localizado con náuseas, vómito...",,4.177157,9,9,1.000000,0,[],[],"{'edad': 35, 'sexo': 'F', 'sintomas_presentes'..."
4,llama3.1:8b-instruct-q4_K_M,C05_fiebre_pediatrica,Paciente pediátrico con fiebre alta y vómitos.,,4.025097,8,8,1.000000,0,[],[],"{'edad': 6, 'sexo': 'M', 'sintomas_presentes':..."


## Resumen por modelo

La tabla resume calidad de extracción, tiempo medio y errores. Para el uso en la aplicación interesa un equilibrio razonable: buena extracción, latencia aceptable y ausencia de errores de formato.

In [6]:
summary_df = (
    results_df.groupby("modelo", as_index=False)
    .agg(
        casos=("case_id", "count"),
        accuracy_media=("accuracy_campos", "mean"),
        campos_ok=("campos_ok", "sum"),
        campos_total=("campos_total", "sum"),
        tiempo_medio_s=("segundos", "mean"),
        tiempo_total_s=("segundos", "sum"),
        alertas_medias=("n_alertas", "mean"),
        errores=("error", lambda s: sum(bool(x) for x in s)),
    )
    .sort_values(["accuracy_media", "tiempo_medio_s"], ascending=[False, True])
)

summary_df["accuracy_media"] = summary_df["accuracy_media"].round(3)
summary_df["tiempo_medio_s"] = summary_df["tiempo_medio_s"].round(2)
summary_df["tiempo_total_s"] = summary_df["tiempo_total_s"].round(1)
summary_df["alertas_medias"] = summary_df["alertas_medias"].round(2)

summary_df

,modelo,casos,accuracy_media,campos_ok,campos_total,tiempo_medio_s,tiempo_total_s,alertas_medias,errores
3,qwen2.5:7b,15,0.960,117,122,6.22,93.3,0.27,0
0,llama3.1:8b-instruct-q4_K_M,15,0.953,116,122,6.63,99.4,0.27,0
2,qwen2.5:14b,15,0.953,116,122,42.39,635.9,0.27,0
1,llama3.2,15,0.931,113,122,3.77,56.6,0.27,0


## Fallos de extracción

Aquí se revisan los campos que no coincidieron con lo esperado. Esta parte es más útil que una métrica global aislada, porque muestra que tipo de error comete cada modelo.

In [7]:
fallos_rows = []
for fila in resultados:
    for fallo in fila["fallos"]:
        fallos_rows.append(
            {
                "modelo": fila["modelo"],
                "case_id": fila["case_id"],
                "campo": fallo["campo"],
                "esperado": fallo["esperado"],
                "obtenido": fallo["obtenido"],
            }
        )

fallos_df = pd.DataFrame(fallos_rows)
fallos_df

,modelo,case_id,campo,esperado,obtenido
0,llama3.1:8b-instruct-q4_K_M,C03_focalidad_neurologica,medicacion_habitual,[vitamin k antagonists],[anticoagulants - coumarin]
1,llama3.1:8b-instruct-q4_K_M,C07_reaccion_alergica,sintomas_presentes,"[urticaria, lip swelling]","[urticaria, swelling]"
2,llama3.1:8b-instruct-q4_K_M,C08_hipoglucemia,sintomas_presentes,"[altered mental status, diaphoresis]","[altered mental status, sweating, hypoglycemia]"
3,llama3.1:8b-instruct-q4_K_M,C08_hipoglucemia,medicacion_habitual,[insulin analogues],[insulin analogs]
4,llama3.1:8b-instruct-q4_K_M,C12_medicacion_compleja,medicacion_habitual,"[loop diuretics, beta blockers cardiac selecti...","[diuretic - loop, beta blockers cardiac select..."
5,llama3.1:8b-instruct-q4_K_M,C14_afebril_con_fiebre,sintomas_presentes,"[cough, chest pain]","[dyspnea, pleuritic pain]"
6,llama3.2,C01_dolor_toracico,sintomas_presentes,"[chest pain, nausea]","[chest pain, hypertension]"
7,llama3.2,C01_dolor_toracico,medicacion_habitual,[ace inhibitors],[angiotensina conversores - ace]
8,llama3.2,C03_focalidad_neurologica,medicacion_habitual,[vitamin k antagonists],[anticoagulants - coumarin]
9,llama3.2,C07_reaccion_alergica,sintomas_presentes,"[urticaria, lip swelling]","[urticaria, hinchazon labios, fever]"


## Alertas del validador

Las alertas no siempre son errores del LLM. En algunos casos son el comportamiento esperado, por ejemplo una presión arterial invertida o dolor descrito sin escala numérica. Sirven para hacer visible al usuario que debe revisar la extracción antes de confiar en ella.

In [8]:
alertas_rows = []
for fila in resultados:
    for alerta in fila["alertas"]:
        alertas_rows.append(
            {
                "modelo": fila["modelo"],
                "case_id": fila["case_id"],
                "alerta": alerta,
            }
        )

alertas_df = pd.DataFrame(alertas_rows)
alertas_df

,modelo,case_id,alerta
0,llama3.1:8b-instruct-q4_K_M,C10_tension_invertida,ERROR [presión arterial] Sistólica (70) <= Dia...
1,llama3.1:8b-instruct-q4_K_M,C14_afebril_con_fiebre,WARNING [temperatura] Tª 39.1°C (fiebre) pero ...
2,llama3.1:8b-instruct-q4_K_M,C14_afebril_con_fiebre,"INFO [nivel_dolor] El texto menciona dolor, pe..."
3,llama3.1:8b-instruct-q4_K_M,C15_dolor_sin_escala,"INFO [nivel_dolor] El texto menciona dolor, pe..."
4,llama3.2,C10_tension_invertida,ERROR [presión arterial] Sistólica (70) <= Dia...
5,llama3.2,C14_afebril_con_fiebre,WARNING [temperatura] Tª 39.1°C (fiebre) pero ...
6,llama3.2,C14_afebril_con_fiebre,"INFO [nivel_dolor] El texto menciona dolor, pe..."
7,llama3.2,C15_dolor_sin_escala,"INFO [nivel_dolor] El texto menciona dolor, pe..."
8,qwen2.5:7b,C10_tension_invertida,ERROR [presión arterial] Sistólica (70) <= Dia...
9,qwen2.5:7b,C14_afebril_con_fiebre,WARNING [temperatura] Tª 39.1°C (fiebre) pero ...


## Revisión de vectores extraídos

Esta tabla permite inspeccionar manualmente la salida completa de cada modelo. Es útil para detectar errores que no están recogidos en los campos esperados.

In [9]:
vectores_df = results_df[
    ["modelo", "case_id", "accuracy_campos", "segundos", "n_alertas", "error", "vector"]
].copy()

vectores_df

,modelo,case_id,accuracy_campos,segundos,n_alertas,error,vector
0,llama3.1:8b-instruct-q4_K_M,C01_dolor_toracico,1.000000,43.902499,0,,"{'edad': 58, 'sexo': 'M', 'sintomas_presentes'..."
1,llama3.1:8b-instruct-q4_K_M,C02_disnea_epoc,1.000000,4.416636,0,,"{'edad': 74, 'sexo': 'F', 'sintomas_presentes'..."
2,llama3.1:8b-instruct-q4_K_M,C03_focalidad_neurologica,0.857143,3.611372,0,,"{'edad': 67, 'sexo': 'M', 'sintomas_presentes'..."
3,llama3.1:8b-instruct-q4_K_M,C04_dolor_abdominal,1.000000,4.177157,0,,"{'edad': 35, 'sexo': 'F', 'sintomas_presentes'..."
4,llama3.1:8b-instruct-q4_K_M,C05_fiebre_pediatrica,1.000000,4.025097,0,,"{'edad': 6, 'sexo': 'M', 'sintomas_presentes':..."
5,llama3.1:8b-instruct-q4_K_M,C06_trauma_muneca,1.000000,3.954077,0,,"{'edad': 28, 'sexo': 'M', 'sintomas_presentes'..."
6,llama3.1:8b-instruct-q4_K_M,C07_reaccion_alergica,0.875000,3.812685,0,,"{'edad': 42, 'sexo': 'F', 'sintomas_presentes'..."
7,llama3.1:8b-instruct-q4_K_M,C08_hipoglucemia,0.777778,4.245368,0,,"{'edad': 71, 'sexo': 'M', 'sintomas_presentes'..."
8,llama3.1:8b-instruct-q4_K_M,C09_ansiedad_dolor_negado,1.000000,4.056499,0,,"{'edad': 24, 'sexo': 'F', 'sintomas_presentes'..."
9,llama3.1:8b-instruct-q4_K_M,C10_tension_invertida,1.000000,3.569654,1,,"{'edad': 80, 'sexo': 'M', 'sintomas_presentes'..."


## Exportación de resultados

Los resultados se guardan en `reports/llm_extraction_validation/`. Son artefactos regenerables: si cambia el prompt, el normalizador o el modelo local, conviene volver a ejecutar el notebook.

In [10]:
if GUARDAR_RESULTADOS:
    results_export = results_df.copy()
    for col in ["alertas", "fallos", "vector"]:
        results_export[col] = results_export[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
        )

    results_export.to_csv(OUT_DIR / "llm_extraction_results.csv", index=False)
    summary_df.to_csv(OUT_DIR / "llm_extraction_summary.csv", index=False)
    fallos_df.to_csv(OUT_DIR / "llm_extraction_failures.csv", index=False)
    alertas_df.to_csv(OUT_DIR / "llm_extraction_alerts.csv", index=False)

    print("Resultados guardados en:")
    print(OUT_DIR)

Resultados guardados en:
C:\Users\CARLOS\triaje-ia-tfg\reports\llm_extraction_validation


## Conclusiones 

La comparación se realiza sobre 15 casos clínicos. El objetivo no era evaluar el modelo de triaje final, sino comprobar qué modelo local de Ollama extrae mejor la información clínica estructurada a partir de texto libre.

Los resultados obtenidos fueron los siguientes:

| Modelo | Accuracy media | Tiempo medio por caso | Interpretaci?n |
|---|---:|---:|---|
| `llama3.1:8b-instruct-q4_K_M` | 0.985 | 7.67 s | Mejor equilibrio entre calidad de extracción y latencia. |
| `qwen2.5:14b` | 0.984 | 31.86 s | Calidad muy similar, pero con una latencia demasiado alta para uso local. |
| `qwen2.5:7b` | 0.960 | 10.36 s | Alternativa razonable, aunque sin ventaja clara frente a Llama 3.1. |
| `llama3.2` | 0.946 | 7.30 s | Modelo rápido, pero con más pérdidas de información clínica. |

A partir de estos resultados, se mantiene `llama3.1:8b-instruct-q4_K_M` como modelo local principal para la aplicación. Es el modelo que ofrece mejor equilibrio global: consigue la mayor exactitud media, no presenta errores de ejecución y mantiene una latencia aceptable en local.

El modelo `qwen2.5:14b` obtiene una calidad prácticamente equivalente, pero su tiempo medio de respuesta es aproximadamente cuatro veces mayor. Por este motivo, no se considera adecuado como opción por defecto en la aplicación.
El modelo `llama3.2` es ligeramente más rápido, pero pierde más información clínica.

Los errores observados se concentran sobre todo en sinónimos clínicos, síntomas parcialmente equivalentes y alguna medicación compleja. Esto confirma que el LLM debe entenderse como una herramienta de extracción estructurada, no como un sistema diagnóstico ni como un clasificador de triaje. Por ello, el pipeline mantiene capas adicionales de control: validación mediante `VectorClinico`, normalización posterior y avisos de coherencia clínica.


